In [1]:
import os
from agent.task_queue import enqueue_task, start_worker, read_task_events

# start_worker()

REDIS_URL = os.getenv("REDIS_URL", "redis://localhost:6379/0")
TASK_STREAM = "research:tasks"
EVENT_STREAM_PREFIX = "research:events"
CONSUMER_GROUP = "research-workers"

# 单个任务的事件流最大保留条数，防止无限增长
EVENT_STREAM_MAXLEN = 500
# XREAD 阻塞超时（毫秒），避免空轮询
STREAM_BLOCK_MS = 5000


In [3]:

import redis
from loguru import logger


async def _get_redis() -> redis.Redis:
    """延迟获取 Redis 连接（复用同一个连接池）."""
    _redis = redis.from_url(
        REDIS_URL,
        decode_responses=True,
        socket_connect_timeout=2.0,  # ← 新增
        socket_timeout=10.0,  # ← 新增
        retry_on_timeout=True,  # ← 新增
        health_check_interval=30,  # ← 新增
    )
    return _redis


r = await _get_redis()


## 消费者组
```bash
XINFO GROUPS research-workers

XINFO GROUPS  BUSYGROUP

```
XINFO STREAM research:tasks
```
```
XINFO CONSUMERS research:tasks research-workers
XPENDING research:tasks research-workers
```
```


In [ ]:
"""创建 Consumer Group（如果不存在）."""
try:
    await r.xgroup_create(TASK_STREAM, CONSUMER_GROUP, id="0", mkstream=True)
    logger.info(f"[TaskQueue] Consumer Group '{CONSUMER_GROUP}' 已创建")
except redis.ResponseError as e:
    if "BUSYGROUP" in str(e):
        logger.debug(f"[TaskQueue] Consumer Group 已存在，跳过创建")
    else:
        raise

In [ ]:
result = await r.xreadgroup(
      CONSUMER_GROUP,
      "worker-1",  # 消费者名称（多实例时每实例一个唯一名）
      {TASK_STREAM: ">"},
      block=STREAM_BLOCK_MS,
      count=1,
  )
if not result: